In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
df = pd.read_csv("data/train.csv")
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin", "SibSp"])
X = df.drop(columns="Survived")
y = df["Survived"]

num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),                 # критично для MLP
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("prep", preprocess),
    ("mlp",  MLPClassifier(
        hidden_layer_sizes=(32, 16),      # 2 скрытых слоя, активация relu между ними
        activation="relu",
        solver="adam",
        alpha=1e-3,                        # L2-регуляризация
        learning_rate_init=1e-3,
        batch_size=32,
        max_iter=500,
        early_stopping=True,               # 10% train уходит на val
        validation_fraction=0.2,
        n_iter_no_change=20,
        random_state=42,
    )),
])
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("Test AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("Итераций:", model.named_steps["mlp"].n_iter_)
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0      0.764     0.882     0.819       110
           1      0.750     0.565     0.645        69

    accuracy                          0.760       179
   macro avg      0.757     0.724     0.732       179
weighted avg      0.758     0.760     0.752       179

Test AUC: 0.8343
Итераций: 29
[[97 13]
 [30 39]]
